# 01 — Greek Mythology Dataset Exploration

**Goal:** Download, audit, clean, and subset an English-language mythology dataset for a reproducible personality-archetype project. This notebook does **not** treat mythological character descriptions as scientifically validated Big Five data.

**Third-party dataset:** [katkaypettitt/greek-gods](https://github.com/katkaypettitt/greek-gods) (MIT); built from Wikipedia and described by its author as containing 445 divine figures. The CSV is downloaded when this notebook is run; a network connection is needed on the first run.

**Expected environment:** Launch Jupyter from repository root (`jupyter lab`) or set the Python working directory to the extracted repository before running this notebook.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if not (ROOT / "src").exists():
    raise RuntimeError("Open Jupyter in the repository root, or change ROOT to the repository directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_processing import load_raw_dataset, clean_dataset, prepare_dataset, SELECTED_NAMES

print("Repository:", ROOT)

## 1. Load and understand the raw dataset

`load_raw_dataset()` caches the source CSV in `data/raw`. It does not scrape Wikipedia or alter the source.

In [ ]:
raw = load_raw_dataset()
print("Raw shape:", raw.shape)
display(raw.head(10))
print("Columns:", raw.columns.tolist())

In [ ]:
audit = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "missing": raw.isna().sum(),
    "unique": raw.nunique(dropna=True)
})
display(audit)
print("Duplicated English names:", raw["name-english"].astype("string").str.strip().duplicated().sum())

## 2. Clean and describe figures

Whitespace is removed from text columns; rows with no English name and duplicated English names are removed.

In [ ]:
gods = clean_dataset(raw)
print("Cleaned shape:", gods.shape)
display(gods["main-type"].value_counts(dropna=False).rename_axis("type").to_frame("count"))
display(gods["sub-type"].value_counts(dropna=False).head(15))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
gods["main-type"].value_counts().plot(kind="bar", ax=ax)
ax.set(title="Greek mythology dataset by main type", xlabel="Category", ylabel="Number of figures")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

## 3. Select the eight MVP archetypes

Our MVP uses a balanced *curated selection* of familiar mythological figures. Descriptions are provenance information only, **not measured personality traits**.

In [ ]:
clean, mvp = prepare_dataset()
assert set(mvp["name-english"]) == set(SELECTED_NAMES)
display(mvp[["name-english", "sub-type", "description"]].sort_values("name-english"))
print("Saved:", ROOT / "data/processed/greek_gods_clean.csv")
print("Saved:", ROOT / "data/processed/mvp_gods.csv")

## 4. Inspect the explicitly *human-authored* archetype profiles

These scores are illustrative. Future work should link each interpretation to mythological evidence and assess sensitivity to alternative annotations.

In [ ]:
import json
profiles = json.loads((ROOT / "data/archetype_profiles.json").read_text(encoding="utf-8"))
profile_table = pd.DataFrame({"name": p["name"], **p["traits"]} for p in profiles).set_index("name")
display(profile_table)
fig, ax = plt.subplots(figsize=(9, 5))
heatmap = ax.imshow(profile_table.values, vmin=0, vmax=1, aspect="auto", cmap="viridis")
ax.set_yticks(range(len(profile_table.index)), profile_table.index)
ax.set_xticks(range(len(profile_table.columns)), profile_table.columns, rotation=30, ha="right")
fig.colorbar(heatmap, ax=ax, label="Human-designed illustrative value")
ax.set_title("Illustrative archetype vectors — NOT psychological measurements")
plt.tight_layout()
plt.show()

## 5. Next experiments

1. Collect and cite multiple passages supporting each narrative archetype.
2. Have independent mythology readers assign trait annotations and quantify disagreement.
3. Run scoring and matching tests using `pytest -q`.
4. Compare distance metrics and how small perturbations alter the ranking.
5. Build an evidence-grounded RAG system to answer English mythology questions.

**Reproducibility:** The dataset and source URLs are documented in the README; all numeric mappings are stored separately in JSON rather than inferred from descriptions.